# SIT320 Advanced Algorithms
## Credit Task: Advanced Hashing and Sorting, code

**Michael Pappas**



## Task 2: 

I worked the Task 2 trace by hand first. This is the simulator I used to check it. It replays both hash schemes with all 17 insertions, the duplicated 26 and 47 included as their own records, and reports the final shape, which is where the comparison table in the report comes from.

In [1]:
BUCKET_SIZE = 3
VALUES = [16, 22, 26, 20, 3, 1, 12, 91, 28, 26, 47, 11, 13, 19, 38, 47, 46]
MAX_GD = 8

def h_ascii(k): return sum(ord(c) for c in str(k))
def h_int(k):   return k

class Bucket:
    def __init__(self, ld):
        self.ld = ld; self.keys = []; self.overflow = []
    def label(self):
        s = ",".join(map(str, self.keys))
        o = f" +OVF[{','.join(map(str,self.overflow))}]" if self.overflow else ""
        return f"[{s}]{o} d'={self.ld}"

class EH:
    def __init__(self, hashfn, gd=1):
        self.h = hashfn; self.gd = gd
        self.dir = [Bucket(gd) for _ in range(2**gd)]

    def idx(self, k): return self.h(k) & ((1 << self.gd) - 1)

    def snapshot(self):
        ids = {}
        for i, b in enumerate(self.dir): ids.setdefault(id(b), []).append(i)
        out = []
        for i, b in enumerate(self.dir):
            shared = " (shared)" if len(ids[id(b)]) > 1 else ""
            out.append(f"    {i:0{self.gd}b} -> {b.label()}{shared}")
        return "\n".join(out)

    def insert(self, k):
        note = []
        # Duplicates are stored as their own records. A duplicate carries the
        # identical hash to its first copy, so no split can separate the pair.
        allkeys = set()
        for b in self.dir: allkeys |= set(b.keys) | set(b.overflow)
        if k in allkeys:
            note.append(f"duplicate: {k} stored as its own record (same hash as the first copy)")
        while True:
            b = self.dir[self.idx(k)]
            if len(b.keys) < BUCKET_SIZE:
                b.keys.append(k); return note
            # would a split separate anything?
            hs = {self.h(x) for x in b.keys} | {self.h(k)}
            if len(hs) == 1 or self.gd >= MAX_GD:
                b.overflow.append(k)
                note.append(f"UNSPLITTABLE: {k} and {b.keys} all hash to {self.h(k)}.")
                note.append("  Splitting cannot separate them, so an OVERFLOW PAGE is chained.")
                return note
            if b.ld == self.gd:
                note.append(f"overflow, d'==d ({self.gd}) -> DOUBLE DIRECTORY, d: {self.gd} -> {self.gd+1}")
                self.dir = self.dir + self.dir[:]; self.gd += 1
            else:
                note.append(f"overflow, d'({b.ld}) < d({self.gd}) -> split bucket only, directory unchanged")
            old = b.keys[:]; bit = 1 << b.ld
            b0, b1 = Bucket(b.ld+1), Bucket(b.ld+1)
            for j in range(len(self.dir)):
                if self.dir[j] is b: self.dir[j] = b1 if (j & bit) else b0
            for key in old: (b1 if (self.h(key) & bit) else b0).keys.append(key)
            note.append(f"  rehash on bit {b.ld}: {old} -> 0-side {b0.keys}, 1-side {b1.keys}"
                        + ("   [uneven split, no space gained]" if not b0.keys or not b1.keys else ""))

# Quick check: replay both schemes and report the final shape.
import io, contextlib
for label, fn in [("Scheme A (ASCII sum)", h_ascii), ("Scheme B (plain int)", h_int)]:
    t = EH(fn); splits = doubles = 0
    for v in VALUES:
        note = t.insert(v)
        doubles += sum('DOUBLE' in n for n in note)
        splits  += sum('rehash on bit' in n for n in note)
    seen, buckets = set(), []
    for b in t.dir:
        if id(b) not in seen:
            seen.add(id(b)); buckets.append(b)
    stored = sum(len(b.keys) + len(b.overflow) for b in buckets)
    print(f"{label:22} d={t.gd}  dir={2**t.gd:3}  buckets={len(buckets)}  "
          f"empty={sum(1 for b in buckets if not b.keys)}  splits={splits}  "
          f"doublings={doubles}  records={stored}  overflow={sum(len(b.overflow) for b in buckets)}")


Scheme A (ASCII sum)   d=4  dir= 16  buckets=9  empty=1  splits=7  doublings=3  records=17  overflow=1
Scheme B (plain int)   d=4  dir= 16  buckets=8  empty=0  splits=6  doublings=3  records=17  overflow=0


---
# Task 3: Extendible Hashing, Code Exploration

## (a) Execution evidence

### The provided implementation



In [2]:
# Provided implementation, exactly as supplied in eHashing.ipynb.

import numpy as np
import os
import string
import random
import csv

def hash_funtion(key):
    
    return  '{0:016b}'.format(key)

class Bucket:
    
    def __init__(self,local_depth,index,empty_spaces,id):
        
        self.id = id
        self.local_depth = local_depth
        self.index = index
        self.empty_spaces = empty_spaces

class Directory:
    
    def  __init__(self,global_depth,directory_records):
        
        self.global_depth = global_depth,
        self.directory_records = directory_records

class DirectoryRecord:
    
    def __init__(self,bucket, hash_prefix):
        
        self.hash_prefix = hash_prefix
        self.value = bucket

bucket_capacity = 2
bucket_number = 3
global_depth = 1

# Initialization of buckets
bucket1 = Bucket(local_depth = 1, empty_spaces = bucket_capacity, index = [], id = 1)
bucket2 = Bucket(local_depth = 1, empty_spaces = bucket_capacity, index = [], id = 2)

# Initialization of directory
directory_records = list()  

directory_records.append(DirectoryRecord(hash_prefix = 0, bucket = bucket1))
directory_records.append(DirectoryRecord(hash_prefix = 1, bucket = bucket2))

directory = Directory(global_depth = 1, directory_records = directory_records)

def insert(index):
    
    global directory
    global bucket_number
    
    t_id = index[0]
    hash_key = hash_funtion(int(t_id))
    
    hash_prefix = int(hash_key[-directory.global_depth[0]:], 2)

    bucket = directory.directory_records[hash_prefix].value
    bucket.index.append(index)
    bucket.empty_spaces = int(bucket.empty_spaces)-1

    if(bucket.empty_spaces < 0):

        tempopary_memory = bucket.index    
        bucket.empty_spaces = bucket_capacity
        bucket.index = []

        if (directory.global_depth[0] > bucket.local_depth):

            # NUMBER OF LINKED BUCKETS
            number_of_links = 2**(directory.global_depth[0] - bucket.local_depth)
            bucket.local_depth = bucket.local_depth + 1
            number_of_modify_links = number_of_links/2 

            new_bucket = Bucket(local_depth = bucket.local_depth, index=[], empty_spaces = bucket_capacity, id = bucket_number)

            for directory_record in directory.directory_records:

                if(directory_record.value == bucket):
                    if(number_of_modify_links != 0):
                        number_of_modify_links = number_of_modify_links - 1
                    else:
                        directory_record.value = new_bucket
                        bucket_number = bucket_number + 1

            for i in range(len(tempopary_memory)):
                insert(tempopary_memory[i])
                

        elif (directory.global_depth[0] == bucket.local_depth):
            
            new_directory_len = 2 * len(directory.directory_records)
            new_directory_records = []

            for directory_record_number in range(new_directory_len):
                new_directory_records.append(DirectoryRecord(hash_prefix=directory_record_number,bucket=Bucket(local_depth=1,index=[],empty_spaces=bucket_capacity,id=bucket_number)))
                bucket_number = bucket_number + 1
            
            new_directory = Directory(global_depth=directory.global_depth[0]+1,directory_records=new_directory_records)

            # REHASING

            for directory_record in directory.directory_records:
                haskey1 = '0'+hash_funtion(directory_record.hash_prefix)
                haskey2 = '1'+hash_funtion(directory_record.hash_prefix)
                new_index1 = int(haskey1[-directory.global_depth[0]:],2)
                new_index2 = int(haskey2[-directory.global_depth[0]:],2)

                new_directory.directory_records[new_index1].value = directory_record.value
                new_directory.directory_records[new_index2].value = directory_record.value

            directory= new_directory

            for i in range(len(tempopary_memory)):

                insert(tempopary_memory[i],lock)
    


### Run 1: the example supplied with the code

A single record never overflows, so no split or doubling is triggered and the code appears to work.

In [3]:
insert([0, 100, 'David'])
print("inserted 1 record, no overflow triggered")
print("global_depth:", directory.global_depth)
print("note the trailing comma in Directory.__init__ made this a TUPLE, not an int,")
print("which is why every other line has to write directory.global_depth[0]")

inserted 1 record, no overflow triggered
global_depth: (1,)
note the trailing comma in Directory.__init__ made this a TUPLE, not an int,
which is why every other line has to write directory.global_depth[0]


### Run 2: the Task 2 data

Forcing an overflow exposes **bug 1** immediately.

In [4]:
VALUES = [16, 22, 26, 20, 3, 1, 12, 91, 28, 26, 47, 11, 13, 19, 38, 47, 46]

# Reset to a clean starting state.
bucket_capacity, bucket_number = 3, 3
b1 = Bucket(local_depth=1, empty_spaces=bucket_capacity, index=[], id=1)
b2 = Bucket(local_depth=1, empty_spaces=bucket_capacity, index=[], id=2)
directory = Directory(global_depth=1, directory_records=[
    DirectoryRecord(hash_prefix=0, bucket=b1),
    DirectoryRecord(hash_prefix=1, bucket=b2)])

for i, v in enumerate(VALUES, 1):
    try:
        insert([v, 100, 'user'])
        print(f"{i:>2}. insert {v:>2} -> ok")
    except Exception as e:
        print(f"{i:>2}. insert {v:>2} -> {type(e).__name__}: {e}")
        print("\nBUG 1: the doubling branch calls insert(record, lock).")
        print("`lock` is not defined anywhere, and insert() takes one argument.")
        print("This branch runs the first time the directory has to double.")
        break

 1. insert 16 -> ok
 2. insert 22 -> ok
 3. insert 26 -> ok
 4. insert 20 -> NameError: name 'lock' is not defined

BUG 1: the doubling branch calls insert(record, lock).
`lock` is not defined anywhere, and insert() takes one argument.
This branch runs the first time the directory has to double.


### Run 3: after patching bug 1

Fixing that one line lets the code run to completion without reporting anything. This shows **bug 2**.

In [5]:
import inspect

# Bug 1 is a single line inside the doubling branch. Instead of pasting the whole
# function again, take the source of the provided insert(), fix that one line, and
# re-execute it. What runs below is the supplied code with exactly one edit.
source = inspect.getsource(insert)
patched = source.replace("insert(tempopary_memory[i],lock)",
                         "insert(tempopary_memory[i])")
assert patched != source, "the patch did not apply"
exec(patched, globals())
print("patched: insert(tempopary_memory[i],lock) -> insert(tempopary_memory[i])")
print(f"lines changed: 1 of {source.count(chr(10))} total")
print()

# Reset to a clean starting state, then run the Task 2 keys through.
bucket_capacity, bucket_number = 3, 3
b1 = Bucket(local_depth=1, empty_spaces=bucket_capacity, index=[], id=1)
b2 = Bucket(local_depth=1, empty_spaces=bucket_capacity, index=[], id=2)
directory = Directory(global_depth=1, directory_records=[
    DirectoryRecord(hash_prefix=0, bucket=b1),
    DirectoryRecord(hash_prefix=1, bucket=b2)])

for v in [16, 22, 26, 20, 3, 1, 12, 91, 28, 47, 11, 13, 19, 38, 46]:
    insert([v, 100, 'user'])

d = directory.global_depth[0]
distinct = len({id(r.value) for r in directory.directory_records})
print(f"global depth      : {d}")
print(f"directory entries : {len(directory.directory_records)}")
print(f"distinct buckets  : {distinct}")
print(f"local depths      : {[r.value.local_depth for r in directory.directory_records]}")
print()
print("BUG 2: there are as many buckets as directory entries, so no two entries")
print("share a bucket. Directory entries are meant to share a bucket when d' < d.")
print("The local depths are also mostly 1, which cannot be right at global depth 4.")

patched: insert(tempopary_memory[i],lock) -> insert(tempopary_memory[i])
lines changed: 1 of 69 total

global depth      : 4
directory entries : 16
distinct buckets  : 16
local depths      : [2, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

BUG 2: there are as many buckets as directory entries, so no two entries
share a bucket. Directory entries are meant to share a bucket when d' < d.
The local depths are also mostly 1, which cannot be right at global depth 4.


### Checking whether the records can be retrieved

A correct lookup takes the last `d` bits of the hash and reads that one directory entry.

In [6]:
def lookup(key):
    """Textbook extendible hashing lookup: last d bits select the directory entry."""
    idx = int(hash_funtion(key)[-d:], 2)
    return any(rec[0] == key for rec in directory.directory_records[idx].value.index), idx


tested = [16, 22, 26, 20, 3, 1, 12, 91, 28, 47, 11, 13, 19, 38, 46]
lost = []
for v in tested:
    found, idx = lookup(v)
    if not found:
        actual = [i for i, r in enumerate(directory.directory_records)
                  if any(x[0] == v for x in r.value.index)]
        lost.append((v, idx, actual))

print(f"{len(lost)} of {len(tested)} keys cannot be found by a correct lookup:\n")
for v, expected, actual in lost:
    print(f"  key {v:>2} (last {d} bits = {v & ((1 << d) - 1):0{d}b}) "
          f"should be at entry {expected:>2}, but is stored at entry {actual}")
print("\nThese records were inserted successfully and are still in memory.")
print("No correct lookup reaches them, and the index raises no error.")

6 of 15 keys cannot be found by a correct lookup:

  key 22 (last 4 bits = 0110) should be at entry  6, but is stored at entry [2]
  key 26 (last 4 bits = 1010) should be at entry 10, but is stored at entry [2]
  key 12 (last 4 bits = 1100) should be at entry 12, but is stored at entry [4]
  key 28 (last 4 bits = 1100) should be at entry 12, but is stored at entry [4]
  key 47 (last 4 bits = 1111) should be at entry 15, but is stored at entry [7]
  key 13 (last 4 bits = 1101) should be at entry 13, but is stored at entry [5]

These records were inserted successfully and are still in memory.
No correct lookup reaches them, and the index raises no error.


### Bug 3: 

In [7]:
print("hash_funtion returns a fixed 16-bit string, so '0' + it is 17 characters.")
print("The slice then takes the last d characters, which never reaches the prepended bit.\n")

old_d = 1  # global depth at the first doubling
for prefix in [0, 1]:
    h1 = '0' + hash_funtion(prefix)
    h2 = '1' + hash_funtion(prefix)
    i1, i2 = int(h1[-old_d:], 2), int(h2[-old_d:], 2)
    print(f"  prefix {prefix}: '0'+h = {h1!r}  ({len(h1)} chars) -> index {i1}")
    print(f"            '1'+h = {h2!r}  ({len(h2)} chars) -> index {i2}")
    print(f"            the two indices collapse to the same entry: {i1 == i2}")

print("\nBUG 3: both halves of the doubled directory map to the same entry, and the")
print("slice uses the old global depth instead of the new one. Half the new directory")
print("keeps the spare empty buckets created a few lines earlier, and the records")
print("in the shared buckets can no longer be reached.")
print("\nThe correct mapping needs no strings at all: old entry p at depth d becomes")
print("entries p and p + 2**d.")
for prefix in [0, 1]:
    print(f"  prefix {prefix} -> entries {prefix} and {prefix + 2 ** old_d}")

hash_funtion returns a fixed 16-bit string, so '0' + it is 17 characters.
The slice then takes the last d characters, which never reaches the prepended bit.

  prefix 0: '0'+h = '00000000000000000'  (17 chars) -> index 0
            '1'+h = '10000000000000000'  (17 chars) -> index 0
            the two indices collapse to the same entry: True
  prefix 1: '0'+h = '00000000000000001'  (17 chars) -> index 1
            '1'+h = '10000000000000001'  (17 chars) -> index 1
            the two indices collapse to the same entry: True

BUG 3: both halves of the doubled directory map to the same entry, and the
slice uses the old global depth instead of the new one. Half the new directory
keeps the spare empty buckets created a few lines earlier, and the records
in the shared buckets can no longer be reached.

The correct mapping needs no strings at all: old entry p at depth d becomes
entries p and p + 2**d.
  prefix 0 -> entries 0 and 2
  prefix 1 -> entries 1 and 3


### Bug 4: 

In [8]:
print(f"{'d':>2} {'d-prime':>8} {'entries sharing the bucket':>28}")
print("-" * 78)
for d_g, d_l in [(2, 1), (3, 1), (4, 2)]:
    matching = [i for i in range(2 ** d_g) if (i & ((1 << d_l) - 1)) == 0]
    half = len(matching) // 2
    ref_keep, ref_move = matching[:half], matching[half:]
    cor_keep = [i for i in matching if not (i & (1 << d_l))]
    cor_move = [i for i in matching if (i & (1 << d_l))]
    verdict = "agrees" if (ref_keep, ref_move) == (cor_keep, cor_move) else "WRONG"
    print(f"{d_g:>2} {d_l:>8} {str(matching):>28}")
    print(f"      reference: keep {ref_keep}, move {ref_move}")
    print(f"      correct  : keep {cor_keep}, move {cor_move}   -> {verdict}")

print("\nBUG 4: correct only when d - d' == 1. The directory entries that share a")
print("bucket are spread evenly through the directory, not bunched into two halves.")

 d  d-prime   entries sharing the bucket
------------------------------------------------------------------------------
 2        1                       [0, 2]
      reference: keep [0], move [2]
      correct  : keep [0], move [2]   -> agrees
 3        1                 [0, 2, 4, 6]
      reference: keep [0, 2], move [4, 6]
      correct  : keep [0, 4], move [2, 6]   -> WRONG
 4        2                [0, 4, 8, 12]
      reference: keep [0, 4], move [8, 12]
      correct  : keep [0, 8], move [4, 12]   -> WRONG

BUG 4: correct only when d - d' == 1. The directory entries that share a
bucket are spread evenly through the directory, not bunched into two halves.


## (b) Code modification

I rewrote the implementation rather than patching six bugs into the structure that caused them. The reasons for each change are in the report.

In [9]:
"""Extendible hashing, rewritten from the implementation provided in eHashing.ipynb.

What I changed:
  * Directory and buckets are wrapped in one class instead of module globals.
  * Bucket contents are a dict keyed by search key, not a list, so lookup,
    duplicate detection and deletion are O(1) instead of a linear scan.
  * The hash function is injectable so different schemes can be compared.
  * Directory doubling and bucket splitting are computed with bit arithmetic
    rather than string slicing.
  * Deletion is supported, with buddy bucket merging and directory halving.
  * A hash value too full to split is detected and given an overflow page,
    instead of doubling the directory forever.
  * A stats() method reports on the shape and health of the structure.
"""


class Bucket:
    """One bucket, which in a real system would be one page on disk."""

    def __init__(self, local_depth):
        self.local_depth = local_depth
        self.records = {}    # key -> record. dict gives O(1) lookup and delete.
        self.overflow = {}   # only used when keys collide beyond any hope of splitting

    def __len__(self):
        return len(self.records) + len(self.overflow)

    def keys(self):
        return list(self.records) + list(self.overflow)

    def __repr__(self):
        body = ",".join(str(k) for k in self.records)
        tail = " +OVF[" + ",".join(str(k) for k in self.overflow) + "]" if self.overflow else ""
        return f"[{body}]{tail} d'={self.local_depth}"


def h_int(key):
    """Plain integer hash: use the key's own binary representation."""
    return key


def h_ascii(key):
    """Sum the ASCII codes of the key rendered as a string."""
    return sum(ord(c) for c in str(key))


class ExtendibleHash:

    def __init__(self, capacity=3, global_depth=1, hash_fn=h_int, verbose=False):
        self.capacity = capacity
        self.global_depth = global_depth
        self.hash_fn = hash_fn
        self.verbose = verbose
        # One distinct bucket per directory entry to start with. Setting
        # global_depth=0 gives a single bucket, which is the other common start state.
        self.directory = [Bucket(global_depth) for _ in range(2 ** global_depth)]
        self.splits = 0
        self.doublings = 0
        self.merges = 0
        self.halvings = 0

    # ---------- internal helpers ----------

    def _log(self, msg):
        if self.verbose:
            print(msg)

    def _index(self, key):
        """Directory entry for a key: the last `global_depth` bits of its hash."""
        return self.hash_fn(key) & ((1 << self.global_depth) - 1)

    def _entries_for(self, bucket):
        return [i for i, b in enumerate(self.directory) if b is bucket]

    def _double_directory(self):
        """Append a copy of the directory. Entry p and entry p + 2^d then point
        at the same bucket, which is exactly the d' < d sharing condition."""
        self.directory = self.directory + self.directory[:]
        self.global_depth += 1
        self.doublings += 1
        self._log(f"    directory doubled, d -> {self.global_depth}")

    def _split(self, bucket):
        """Split one bucket in two and repoint the directory entries that
        referenced it. Entries whose bit at position local_depth is 1 go to the
        new high bucket, the rest stay with the low bucket."""
        bit = 1 << bucket.local_depth
        new_depth = bucket.local_depth + 1
        low, high = Bucket(new_depth), Bucket(new_depth)

        for i in self._entries_for(bucket):
            self.directory[i] = high if (i & bit) else low

        for key, record in bucket.records.items():
            target = high if (self.hash_fn(key) & bit) else low
            target.records[key] = record

        self.splits += 1
        self._log(f"    split on bit {bucket.local_depth}: "
                  f"low={list(low.records)} high={list(high.records)}")
        return low, high

    # ---------- public API ----------

    def search(self, key):
        """Return the record for a key, or None. Costs one bucket read, plus
        one more if the bucket has an overflow page."""
        bucket = self.directory[self._index(key)]
        if key in bucket.records:
            return bucket.records[key]
        return bucket.overflow.get(key)

    def __contains__(self, key):
        return self.search(key) is not None

    def insert(self, key, record=None):
        """Insert a key. Returns True if it was added, False if already present."""
        if record is None:
            record = key
        if key in self:
            self._log(f"  {key}: already present, ignored")
            return False

        while True:
            bucket = self.directory[self._index(key)]

            if len(bucket.records) < self.capacity:
                bucket.records[key] = record
                return True

            # Would a split separate anything? If every key in the bucket has
            # the identical hash, splitting inspects one more bit that is the
            # same for all of them, so the directory would double forever.
            # This is the case an overflow page exists for.
            hashes = {self.hash_fn(k) for k in bucket.records} | {self.hash_fn(key)}
            if len(hashes) == 1:
                bucket.overflow[key] = record
                self._log(f"  {key}: unsplittable collision on hash "
                          f"{self.hash_fn(key)}, chained to an overflow page")
                return True

            if bucket.local_depth == self.global_depth:
                self._double_directory()
            self._split(bucket)

    def delete(self, key):
        """Delete a key, then try to merge the bucket with its buddy."""
        bucket = self.directory[self._index(key)]
        if key in bucket.records:
            del bucket.records[key]
            # Pull a record back out of the overflow page if one is waiting.
            if bucket.overflow:
                k, v = bucket.overflow.popitem()
                bucket.records[k] = v
        elif key in bucket.overflow:
            del bucket.overflow[key]
        else:
            return False

        self._try_merge(bucket)
        self._try_halve()
        return True

    def _try_merge(self, bucket):
        """Merge a bucket with its buddy if their contents fit in one bucket.

        The buddy is reached by flipping bit (local_depth - 1) of any directory
        entry pointing at this bucket, which is the same as saying the two
        share their last (local_depth - 1) bits.
        """
        if bucket.local_depth == 0 or bucket.overflow:
            return
        entries = self._entries_for(bucket)
        if not entries:
            return

        buddy_index = entries[0] ^ (1 << (bucket.local_depth - 1))
        buddy = self.directory[buddy_index]

        if buddy is bucket or buddy.local_depth != bucket.local_depth or buddy.overflow:
            return
        if len(bucket) + len(buddy) > self.capacity:
            return

        merged = Bucket(bucket.local_depth - 1)
        merged.records.update(bucket.records)
        merged.records.update(buddy.records)
        for i in self._entries_for(bucket) + self._entries_for(buddy):
            self.directory[i] = merged
        self.merges += 1
        self._log(f"    merged buddies into {merged}")
        self._try_merge(merged)

    def _try_halve(self):
        """Halve the directory while every local depth is strictly below the
        global depth, which means no entry needs the extra bit."""
        while (self.global_depth > 0
               and all(b.local_depth < self.global_depth for b in self.directory)):
            half = len(self.directory) // 2
            self.directory = self.directory[:half]
            self.global_depth -= 1
            self.halvings += 1
            self._log(f"    directory halved, d -> {self.global_depth}")

    # ---------- reporting ----------

    def buckets(self):
        """Distinct buckets, since several directory entries may share one."""
        seen, out = set(), []
        for b in self.directory:
            if id(b) not in seen:
                seen.add(id(b))
                out.append(b)
        return out

    def stats(self):
        bs = self.buckets()
        stored = sum(len(b) for b in bs)
        return {
            "global_depth": self.global_depth,
            "directory_entries": len(self.directory),
            "distinct_buckets": len(bs),
            "empty_buckets": sum(1 for b in bs if len(b) == 0),
            "keys_stored": stored,
            "capacity_total": len(bs) * self.capacity,
            "occupancy": stored / (len(bs) * self.capacity) if bs else 0.0,
            "overflow_keys": sum(len(b.overflow) for b in bs),
            "splits": self.splits,
            "doublings": self.doublings,
            "merges": self.merges,
            "halvings": self.halvings,
        }

    def show(self):
        counts = {}
        for b in self.directory:
            counts[id(b)] = counts.get(id(b), 0) + 1
        lines = [f"d = {self.global_depth}"]
        for i, b in enumerate(self.directory):
            shared = "  (shared)" if counts[id(b)] > 1 else ""
            lines.append(f"    {i:0{self.global_depth}b} -> {b}{shared}")
        return "\n".join(lines)

    def verify(self):
        """Every stored key must be reachable by a correct lookup, and every
        key must sit in a bucket its hash actually points at."""
        problems = []
        for i, b in enumerate(self.directory):
            for key in b.keys():
                if self._index(key) != i and self.directory[self._index(key)] is not b:
                    problems.append(f"key {key} sits at entry {i} but hashes to {self._index(key)}")
        return problems


### Test 1: reproducing the hand-worked Task 2 trace

In [10]:
VALUES = [16, 22, 26, 20, 3, 1, 12, 91, 28, 26, 47, 11, 13, 19, 38, 47, 46]

for label, fn in [("Scheme B (plain int)", h_int), ("Scheme A (ASCII sum)", h_ascii)]:
    t = ExtendibleHash(capacity=3, global_depth=1, hash_fn=fn)
    for v in VALUES:
        t.insert(v)
    s = t.stats()
    print(f"{label:22} d={s['global_depth']} dir={s['directory_entries']:>2} "
          f"buckets={s['distinct_buckets']} empty={s['empty_buckets']} "
          f"splits={s['splits']} doublings={s['doublings']} keys={s['keys_stored']} "
          f"overflow={s['overflow_keys']} occupancy={s['occupancy']:.0%}")
    print(f"   {t.verify() or 'every key reachable by a correct lookup'}")

print("\nThe rewrite indexes unique keys, so the duplicated 26 and 47 are rejected and 15")
print("distinct keys are stored. The structure still shows the overflow page and the")
print("empty bucket from the hand-worked Scheme A trace.")

Scheme B (plain int)   d=4 dir=16 buckets=8 empty=0 splits=6 doublings=3 keys=15 overflow=0 occupancy=62%
   every key reachable by a correct lookup
Scheme A (ASCII sum)   d=4 dir=16 buckets=8 empty=1 splits=6 doublings=3 keys=15 overflow=1 occupancy=62%
   every key reachable by a correct lookup

The rewrite indexes unique keys, so the duplicated 26 and 47 are rejected and 15
distinct keys are stored. The structure still shows the overflow page and the
empty bucket from the hand-worked Scheme A trace.


### Test 2: retrieving the keys the provided code loses

In [11]:
t = ExtendibleHash(capacity=3, hash_fn=h_int)
for v in VALUES:
    t.insert(v)

unreachable = [v for v in sorted(set(VALUES)) if v not in t]
print("unreachable keys:", unreachable or "none")
print("search(22)  ->", t.search(22))
print("search(26)  ->", t.search(26))
print("search(999) ->", t.search(999), "(absent key returns None rather than raising)")
print()
print(t.show())

unreachable keys: none
search(22)  -> 22
search(26)  -> 26
search(999) -> None (absent key returns None rather than raising)

d = 4
    0000 -> [16] d'=3  (shared)
    0001 -> [1,13] d'=2  (shared)
    0010 -> [26] d'=3  (shared)
    0011 -> [3,19] d'=4
    0100 -> [20,12,28] d'=3  (shared)
    0101 -> [1,13] d'=2  (shared)
    0110 -> [22,38,46] d'=3  (shared)
    0111 -> [47] d'=3  (shared)
    1000 -> [16] d'=3  (shared)
    1001 -> [1,13] d'=2  (shared)
    1010 -> [26] d'=3  (shared)
    1011 -> [91,11] d'=4
    1100 -> [20,12,28] d'=3  (shared)
    1101 -> [1,13] d'=2  (shared)
    1110 -> [22,38,46] d'=3  (shared)
    1111 -> [47] d'=3  (shared)


### Test 3: edge cases (duplicates, empty structure, repeated hash values)

In [12]:
# Duplicate inserts
t = ExtendibleHash()
print("insert 5 three times ->", [t.insert(5) for _ in range(3)], "(True once, then False)")

# Empty structure
t2 = ExtendibleHash()
print("empty structure      ->", t2.stats()['keys_stored'], "keys,",
      t2.verify() or "consistent", "| search(1) ->", t2.search(1))

# The four keys from Task 2 that all hash to the same value
t3 = ExtendibleHash(capacity=3, hash_fn=h_ascii)
for v in [91, 28, 19, 46]:
    t3.insert(v)
print("4 keys all hashing to 106 ->",
      f"stored {sum(len(b) for b in t3.buckets())},",
      f"overflow {t3.stats()['overflow_keys']},",
      f"all retrievable {all(v in t3 for v in [91, 28, 19, 46])}")
print("  the provided code doubles the directory here until it runs out of memory")

insert 5 three times -> [True, False, False] (True once, then False)
empty structure      -> 0 keys, consistent | search(1) -> None
4 keys all hashing to 106 -> stored 4, overflow 1, all retrievable True
  the provided code doubles the directory here until it runs out of memory


### Test 4: varying the number of initial buckets

In [13]:
print(f"{'start d':>8} {'buckets':>8} {'final d':>8} {'splits':>7} {'doublings':>10} {'occupancy':>10}")
print("-" * 56)
for gd in [0, 1, 2, 3]:
    t = ExtendibleHash(capacity=3, global_depth=gd)
    for v in VALUES:
        t.insert(v)
    s = t.stats()
    print(f"{gd:>8} {2 ** gd:>8} {s['global_depth']:>8} {s['splits']:>7} "
          f"{s['doublings']:>10} {s['occupancy']:>9.0%}   {t.verify() or 'consistent'}")

print("\nEvery starting point ends at the same final depth. Starting deeper just does")
print("the work up front that splitting would have done anyway.")

 start d  buckets  final d  splits  doublings  occupancy
--------------------------------------------------------
       0        1        4       7          4       62%   consistent
       1        2        4       6          3       62%   consistent
       2        4        4       4          2       62%   consistent
       3        8        4       1          1       56%   consistent

Every starting point ends at the same final depth. Starting deeper just does
the work up front that splitting would have done anyway.


### Test 5: deletion, buddy merging and directory halving

In [14]:
KEYS = [16, 22, 26, 20, 3, 1, 12, 91, 28, 47, 11, 13, 19, 38, 46]
t = ExtendibleHash(capacity=3, global_depth=1, verbose=True)
for v in KEYS:
    t.insert(v)
t.verbose = True
print(f"after inserts: d={t.global_depth}, buckets={len(t.buckets())}\n")

print("deleting every key, watching buddies merge and the directory halve:")
for v in KEYS:
    t.delete(v)

s = t.stats()
print(f"\nfinal: d={s['global_depth']}, directory={s['directory_entries']}, "
      f"buckets={s['distinct_buckets']}, keys={s['keys_stored']}, "
      f"merges={s['merges']}, halvings={s['halvings']}")
print("verify:", t.verify() or "consistent")
print("\nThe structure collapsed back to a single empty bucket at depth 0, which is")
print("where it started.")

    directory doubled, d -> 2
    split on bit 1: low=[16] high=[22, 26]
    directory doubled, d -> 3
    split on bit 2: low=[16] high=[20, 12]
    split on bit 1: low=[1] high=[3, 91]
    split on bit 2: low=[3, 91] high=[47]
    directory doubled, d -> 4
    split on bit 3: low=[3] high=[91, 11]
    split on bit 2: low=[26] high=[22, 38]
after inserts: d=4, buckets=8

deleting every key, watching buddies merge and the directory halve:
    merged buddies into [20,12,28] d'=2
    merged buddies into [38,46,26] d'=2
    merged buddies into [19,91,11] d'=3
    directory halved, d -> 3
    merged buddies into [28,38,46] d'=1
    merged buddies into [19,11,47] d'=2
    directory halved, d -> 2
    merged buddies into [19,11,13] d'=1
    directory halved, d -> 1
    merged buddies into [19,38,46] d'=0
    directory halved, d -> 0

final: d=0, directory=1, buckets=1, keys=0, merges=7, halvings=4
verify: consistent

The structure collapsed back to a single empty bucket at depth 0, which is


### Test 6: loading from an external file and from generated data

In [15]:
import csv
import numpy as np

# Write then read an external CSV.
with open('keys.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['key', 'amount', 'user'])
    for k in KEYS:
        w.writerow([k, k * 10, f'user{k}'])

t = ExtendibleHash(capacity=4)
with open('keys.csv') as f:
    for row in csv.DictReader(f):
        t.insert(int(row['key']), (int(row['amount']), row['user']))
print(f"loaded {t.stats()['keys_stored']} records from keys.csv")
print("search(38) ->", t.search(38), "|", t.verify() or "consistent")

# Dynamic data from numpy.
rng = np.random.default_rng(7)
keys = rng.integers(0, 100000, size=2000).tolist()
t = ExtendibleHash(capacity=8, global_depth=2)
added = sum(t.insert(int(k)) for k in keys)
s = t.stats()
print(f"\noffered {len(keys)} random keys, added {added} ({len(keys) - added} were duplicates)")
print(f"  d={s['global_depth']} directory={s['directory_entries']} "
      f"buckets={s['distinct_buckets']} occupancy={s['occupancy']:.0%} "
      f"splits={s['splits']} doublings={s['doublings']} overflow={s['overflow_keys']}")
print("  every key retrievable:", all(int(k) in t for k in set(keys)))
print(" ", t.verify() or "consistent")

loaded 15 records from keys.csv
search(38) -> (380, 'user38') | consistent

offered 2000 random keys, added 1981 (19 were duplicates)
  d=10 directory=1024 buckets=353 occupancy=70% splits=349 doublings=8 overflow=0
  every key retrievable: True
  consistent


### Test 7: mixed inserts and deletes

10,000 random operations checked against a Python `set` modelling what should be stored. This is the test I trust most, because it runs splitting and merging against each other instead of testing them on their own.

In [16]:
import random

t = ExtendibleHash(capacity=4, global_depth=1)
model = set()
random.seed(1)

for _ in range(10000):
    k = random.randint(0, 500)
    if random.random() < 0.6:
        t.insert(k)
        model.add(k)
    else:
        t.delete(k)
        model.discard(k)

live = {k for b in t.buckets() for k in b.keys()}
print(f"model holds     : {len(model)} keys")
print(f"structure holds : {len(live)} keys")
print(f"contents match  : {model == live}")
print(f"all retrievable : {all(k in t for k in model)}")
print(f"\nd={t.global_depth} splits={t.splits} doublings={t.doublings} "
      f"merges={t.merges} halvings={t.halvings}")
print(t.verify() or "consistent")
print("\nHalvings is 0 despite hundreds of merges. With this many keys still live,")
print("some bucket always needs the full global depth, so the directory never")
print("qualifies to shrink. Merging buckets and halving the directory are two")
print("separate conditions, and the second one is much harder to meet.")

model holds     : 313 keys


structure holds : 313 keys
contents match  : True
all retrievable : True

d=7 splits=785 doublings=6 merges=683 halvings=0
consistent

Halvings is 0 despite hundreds of merges. With this many keys still live,
some bucket always needs the full global depth, so the directory never
qualifies to shrink. Merging buckets and halving the directory are two
separate conditions, and the second one is much harder to meet.


---
# Task 4: Simulated External Merge Sort

`N = 100` pages, `page_size = 4`, 400 random integers. I/Os are counted inside the `Disk` class so the tally cannot drift from what the code actually does.

In [17]:
import numpy as np
import math
import heapq


class Disk:
    """Simulated disk.

    Every page that crosses the boundary between disk and memory costs exactly
    1 I/O, whichever direction it goes. Counting here rather than in the sort
    means the tally always reflects what the code really did.
    """

    def __init__(self, pages):
        self.pages = {i: list(p) for i, p in enumerate(pages)}
        self.ios = 0

    def read(self, i):
        self.ios += 1
        return list(self.pages[i])

    def write(self, i, records):
        self.ios += 1
        self.pages[i] = list(records)

    def flat(self):
        """All records in page order. Used only for checking the result."""
        return [r for i in sorted(self.pages) for r in self.pages[i]]


# Build the simulated data file: 100 pages of 4 random records each.
N, PAGE_SIZE, B = 100, 4, 5
rng = np.random.default_rng(42)
data = rng.integers(1, 1000, size=(N, PAGE_SIZE)).tolist()

disk = Disk(data)
print(f"{N} pages x {PAGE_SIZE} records = {N * PAGE_SIZE} records")
for i in [0, 1, 2, 99]:
    print(f"  page {i:>2}: {disk.pages[i]}")

100 pages x 4 records = 400 records
  page  0: [90, 774, 654, 439]
  page  1: [433, 858, 86, 697]
  page  2: [202, 95, 526, 975]
  page 99: [45, 197, 281, 311]


### The sort and merge passes

In [18]:
def sort_pass(disk, N, page_size, B, verbose=True):
    """Pass 0, the conquer pass.

    Fill all B buffers, sort those B*page_size records in memory, then write
    them straight back. Each group of B pages becomes one sorted run, so this
    pass costs 2N I/Os and produces ceil(N/B) runs.
    """
    runs = []
    for start in range(0, N, B):
        pages = list(range(start, min(start + B, N)))
        buffer = [r for p in pages for r in disk.read(p)]   # B reads, fills memory
        buffer.sort()                                        # in memory, costs no I/O
        for n, p in enumerate(pages):                        # B writes back
            disk.write(p, buffer[n * page_size:(n + 1) * page_size])
        runs.append(pages)
    if verbose:
        print(f"Pass 0 (sort)  : {len(runs)} sorted runs of up to {B} pages"
              f"          | I/Os so far: {disk.ios}")
    return runs


def merge_runs(source, dest, runs, page_size, out_start):
    """Merge up to B-1 sorted runs into one, using B pages of memory.

    One input buffer holds the current page of each run, and the last buffer
    holds output. A heap picks the smallest unconsumed record across the runs
    in O(log k) rather than scanning all of them.

    The merged output goes to `dest`, a separate region. Writing back over
    `source` would clobber input pages that have not been read yet.
    """
    buffers, pos, next_page = [], [], []
    for run in runs:
        buffers.append(source.read(run[0]))   # one input buffer per run
        pos.append(0)
        next_page.append(1)

    heap = []
    for i in range(len(runs)):
        if buffers[i]:
            heapq.heappush(heap, (buffers[i][0], i))
            pos[i] = 1

    out_buffer, out_page = [], out_start
    while heap:
        value, i = heapq.heappop(heap)
        out_buffer.append(value)

        if len(out_buffer) == page_size:      # output buffer full, flush it
            dest.write(out_page, out_buffer)
            out_page += 1
            out_buffer = []

        if pos[i] == len(buffers[i]):         # input buffer drained, refill it
            if next_page[i] < len(runs[i]):
                buffers[i] = source.read(runs[i][next_page[i]])
                next_page[i] += 1
                pos[i] = 0
            else:
                continue                       # this run is exhausted

        heapq.heappush(heap, (buffers[i][pos[i]], i))
        pos[i] += 1

    if out_buffer:                             # flush the last partial page
        dest.write(out_page, out_buffer)
        out_page += 1

    return list(range(out_start, out_page))


def external_merge_sort(disk, N, page_size, B, verbose=True):
    """Full external merge sort. Returns the number of passes."""
    runs = sort_pass(disk, N, page_size, B, verbose)

    p = 0
    while len(runs) > 1:
        p += 1
        output = Disk([[] for _ in range(N)])
        output.ios = 0

        new_runs, cursor = [], 0
        for g in range(0, len(runs), B - 1):        # B-1 runs at a time
            group = runs[g:g + B - 1]
            new_runs.append(merge_runs(disk, output, group, page_size, cursor))
            cursor += sum(len(r) for r in group)

        disk.ios += output.ios                       # carry the I/O tally forward
        disk.pages = output.pages                    # output region becomes the new input
        runs = new_runs

        if verbose:
            print(f"Pass {p} (merge) : {B - 1}-way merge, {len(runs)} runs remaining"
                  f"           | I/Os so far: {disk.ios}")

    return p + 1

### Running it

In [19]:
expected = sorted(disk.flat())
passes = external_merge_sort(disk, N, PAGE_SIZE, B)
result = disk.flat()

formula = 2 * N * (1 + math.ceil(math.log(N / B, B - 1)))

print(f"\npasses          : {passes}")
print(f"total I/Os      : {disk.ios}")
print(f"formula 2N(1 + ceil(log_(B-1)(N/B))) : {formula}")
print(f"matches formula : {disk.ios == formula}")
print(f"correctly sorted: {result == expected}")
print(f"records         : {len(result)} (expected {N * PAGE_SIZE})")
print(f"\nfirst page: {disk.pages[0]}")
print(f"last page : {disk.pages[N - 1]}")

Pass 0 (sort)  : 20 sorted runs of up to 5 pages          | I/Os so far: 200
Pass 1 (merge) : 4-way merge, 5 runs remaining           | I/Os so far: 400
Pass 2 (merge) : 4-way merge, 2 runs remaining           | I/Os so far: 600
Pass 3 (merge) : 4-way merge, 1 runs remaining           | I/Os so far: 800

passes          : 4
total I/Os      : 800
formula 2N(1 + ceil(log_(B-1)(N/B))) : 800
matches formula : True
correctly sorted: True
records         : 400 (expected 400)

first page: [6, 8, 10, 20]
last page : [991, 992, 992, 996]


### Buffer sweep

If the implementation were quietly holding everything in memory, the buffer size would make no difference to the number of passes.

In [20]:
def trial(B, N=100, page_size=4, seed=42):
    rng = np.random.default_rng(seed)
    data = rng.integers(1, 1000, size=(N, page_size)).tolist()
    d = Disk(data)
    expected = sorted(d.flat())
    passes = external_merge_sort(d, N, page_size, B, verbose=False)
    formula = 2 * N * (1 + math.ceil(math.log(N / B, B - 1)))
    return passes, d.ios, formula, d.flat() == expected


print(f"{'B':>3} {'runs after pass 0':>18} {'passes':>7} {'I/Os':>6} {'formula':>8} {'sorted':>7}")
print("-" * 54)
for B in [3, 4, 5, 8, 11, 21, 51]:
    passes, ios, formula, ok = trial(B)
    print(f"{B:>3} {math.ceil(100 / B):>18} {passes:>7} {ios:>6} {formula:>8} {str(ok):>7}")

  B  runs after pass 0  passes   I/Os  formula  sorted
------------------------------------------------------
  3                 34       7   1400     1400    True
  4                 25       4    800      800    True
  5                 20       4    800      800    True
  8                 13       3    600      600    True
 11                 10       2    400      400    True
 21                  5       2    400      400    True
 51                  2       2    400      400    True
